# TP - Autoencoders Convolucionales
### Redes Neuronales / Deep Learning

**Trabajo en grupos de 2 personas.** Pueden consultarse entre grupos.

**Defensa oral: 40 minutos por grupo**, donde deberán comentar decisiones de diseño, resultados obtenidos, dificultades encontradas y responder las preguntas de reflexión de cada notebook.

Esta es la **Notebook 1 de 3**: Autoencoder Convolucional. Después siguen VAE y GAN.

## Objetivos
- Implementar un autoencoder convolucional para CIFAR-10.
- Combinar una loss de reconstrucción por píxel con una **perceptual loss** basada en una red preentrenada (VGG16).
- Usar **TensorBoard** y **MLflow** para trackear el entrenamiento.
- Usar **Optuna** para buscar hiperparámetros.

## Qué hay que completar
Los bloques marcados con `# TODO` son los que tienen que completar ustedes. El resto del código ya está resuelto para que se puedan enfocar en los conceptos nuevos.


## 0. Setup

En Colab: `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.

In [ ]:
# Instalamos las librerías que no vienen por defecto en Colab
!pip install -q mlflow optuna


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from torchvision.models import vgg16, VGG16_Weights
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)


## 1. Datos: CIFAR-10

Normalizamos a `[0, 1]` porque el decoder va a terminar con una `Sigmoid`.

In [ ]:
transform = transforms.ToTensor()  # deja los pixeles en [0, 1]

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} imágenes | Test: {len(test_dataset)} imágenes")


In [ ]:
# Visualizamos un batch de ejemplo
imgs, _ = next(iter(train_loader))
grid = make_grid(imgs[:16], nrow=8)
plt.figure(figsize=(10, 3))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Ejemplos de CIFAR-10")
plt.show()


## 2. Arquitectura

Vamos a armar un encoder y un decoder convolucionales simétricos.

**Encoder** (entrada `3x32x32`):
- Conv 3→32 canales, kernel 4, stride 2, padding 1  → `32x16x16`
- Conv 32→64 canales, kernel 4, stride 2, padding 1 → `64x8x8`
- Conv 64→128 canales, kernel 4, stride 2, padding 1 → `128x4x4`
- Flatten + `Linear` a `latent_dim`

Usen `nn.ReLU()` como activación después de cada conv (menos en la última capa lineal).

### TODO 1: completar el Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        # TODO: completar las 3 capas convolucionales según la tabla de arriba.
        # Guardá el resultado en self.conv (podés usar nn.Sequential)
        self.conv = nn.Sequential(
            # nn.Conv2d(...), nn.ReLU(),
            # nn.Conv2d(...), nn.ReLU(),
            # nn.Conv2d(...), nn.ReLU(),
        )
        self.fc = nn.Linear(128 * 4 * 4, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # flatten
        z = self.fc(x)
        return z


### TODO 2: completar el Decoder

Es la operación inversa: partimos de `latent_dim`, reconstruimos hasta `3x32x32`.

- `Linear` de `latent_dim` a `128*4*4`, reshape a `(128, 4, 4)`
- ConvTranspose 128→64, kernel 4, stride 2, padding 1 → `64x8x8`  (+ ReLU)
- ConvTranspose 64→32, kernel 4, stride 2, padding 1 → `32x16x16` (+ ReLU)
- ConvTranspose 32→3, kernel 4, stride 2, padding 1 → `3x32x32`  (+ **Sigmoid**, para que quede en `[0,1]`)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        # TODO: completar las 3 capas de ConvTranspose2d según la tabla de arriba.
        self.deconv = nn.Sequential(
            # nn.ConvTranspose2d(...), nn.ReLU(),
            # nn.ConvTranspose2d(...), nn.ReLU(),
            # nn.ConvTranspose2d(...), nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 128, 4, 4)
        x = self.deconv(x)
        return x


class Autoencoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


In [ ]:
# Sanity check rápido: si el Encoder/Decoder están bien armados esto no debería tirar error
# y la forma de salida debe ser igual a la de entrada.
_dummy = torch.randn(4, 3, 32, 32)
_model_check = Autoencoder(latent_dim=128)
_out = _model_check(_dummy)
assert _out.shape == _dummy.shape, f"Shape incorrecta: {_out.shape} != {_dummy.shape}"
print("Shapes OK:", _out.shape)


## 3. Perceptual Loss (VGG16)

La loss de MSE por píxel tiende a dar reconstrucciones borrosas. Una alternativa (o complemento) es comparar las imágenes en el espacio de **features** de una red preentrenada (VGG16), que captura mejor texturas y estructuras.

Usamos solo las primeras capas convolucionales de VGG16 (`features[:8]`), **congeladas** (no se entrenan). No hace falta redimensionar las imágenes: las capas convolucionales aceptan cualquier tamaño espacial.

### TODO 3: completar la Perceptual Loss

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self, layers=8):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:layers]
        # TODO: congelar los parámetros de vgg (que no se actualicen con el optimizer)
        # pista: iterar vgg.parameters() y poner requires_grad = False

        self.vgg = vgg.eval().to(device)
        self.mse = nn.MSELoss()

    def forward(self, x_hat, x):
        # x y x_hat están en [0,1] con shape (B,3,32,32)
        feat_hat = self.vgg(x_hat)
        feat_real = self.vgg(x)
        # TODO: devolver el MSE entre feat_hat y feat_real
        return None  # reemplazar por el cálculo correcto


def combined_loss(x_hat, x, perceptual_loss_fn, lam=0.1):
    '''Loss combinada: MSE de pixeles + lam * perceptual loss.'''
    pixel_loss = nn.functional.mse_loss(x_hat, x)
    # TODO: calcular perc_loss usando perceptual_loss_fn(x_hat, x)
    perc_loss = None
    # TODO: devolver pixel_loss + lam * perc_loss (y también devolver ambos por separado para loguear)
    total = None
    return total, pixel_loss, perc_loss


In [ ]:
# Sanity check de la perceptual loss (requiere internet para bajar los pesos de VGG16 la primera vez)
perceptual_loss_fn = PerceptualLoss()
_a = torch.rand(2, 3, 32, 32).to(device)
_b = torch.rand(2, 3, 32, 32).to(device)
_total, _pix, _perc = combined_loss(_a, _b, perceptual_loss_fn, lam=0.1)
print("total:", _total, "| pixel:", _pix, "| perceptual:", _perc)
assert _total is not None, "Completá los TODO de combined_loss / PerceptualLoss"


## 4. TensorBoard y MLflow

- **TensorBoard**: vamos a loguear la loss por época y una grilla de reconstrucciones cada pocas épocas.
- **MLflow**: vamos a loguear hiperparámetros, métricas por época, y el modelo final como artifact.

Esto ya está resuelto, solo tienen que ejecutarlo y entender qué hace.

In [ ]:
%load_ext tensorboard


In [ ]:
def train_autoencoder(model, loader, optimizer, perceptual_loss_fn, epochs, lam=0.1,
                       writer=None, run_name="ae", log_every=1):
    model.to(device)
    history = []
    for epoch in range(epochs):
        model.train()
        running_total, running_pix, running_perc = 0.0, 0.0, 0.0
        for imgs, _ in loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            x_hat = model(imgs)
            total, pix, perc = combined_loss(x_hat, imgs, perceptual_loss_fn, lam=lam)
            total.backward()
            optimizer.step()
            running_total += total.item() * imgs.size(0)
            running_pix += pix.item() * imgs.size(0)
            running_perc += perc.item() * imgs.size(0)

        n = len(loader.dataset)
        avg_total, avg_pix, avg_perc = running_total / n, running_pix / n, running_perc / n
        history.append(avg_total)
        print(f"[{run_name}] Epoch {epoch+1}/{epochs} - loss_total: {avg_total:.4f} "
              f"(pixel: {avg_pix:.4f}, perceptual: {avg_perc:.4f})")

        if writer is not None:
            writer.add_scalar("loss/total", avg_total, epoch)
            writer.add_scalar("loss/pixel", avg_pix, epoch)
            writer.add_scalar("loss/perceptual", avg_perc, epoch)
        mlflow.log_metric("loss_total", avg_total, step=epoch)
        mlflow.log_metric("loss_pixel", avg_pix, step=epoch)
        mlflow.log_metric("loss_perceptual", avg_perc, step=epoch)

        if writer is not None and epoch % log_every == 0:
            model.eval()
            with torch.no_grad():
                sample, _ = next(iter(loader))
                sample = sample[:8].to(device)
                recon = model(sample)
                comparison = torch.cat([sample, recon])
                grid = make_grid(comparison, nrow=8)
                writer.add_image("originales_vs_reconstruidas", grid, epoch)
    return history


## 5. Búsqueda de hiperparámetros con Optuna

Buscamos `lr` y `latent_dim` optimizando el loss de validación, con pocos trials y pocas épocas (para que sea rápido). Después entrenamos el modelo final con los mejores hiperparámetros por más épocas.

In [ ]:
def evaluate(model, loader, perceptual_loss_fn, lam=0.1):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            x_hat = model(imgs)
            total, _, _ = combined_loss(x_hat, imgs, perceptual_loss_fn, lam=lam)
            total_loss += total.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    latent_dim = trial.suggest_categorical("latent_dim", [32, 64, 128])

    model = Autoencoder(latent_dim=latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_autoencoder(model, train_loader, optimizer, perceptual_loss_fn,
                       epochs=3, writer=None, run_name=f"trial_{trial.number}")
    val_loss = evaluate(model, test_loader, perceptual_loss_fn)
    return val_loss


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor loss de validación:", study.best_value)


## 6. Entrenamiento final

Usamos los mejores hiperparámetros encontrados por Optuna y entrenamos por más épocas, logueando todo a TensorBoard y MLflow.

In [ ]:
best_params = study.best_params
LATENT_DIM = best_params["latent_dim"]
LR = best_params["lr"]
EPOCHS = 20
LAMBDA_PERCEPTUAL = 0.1

model = Autoencoder(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
writer = SummaryWriter(log_dir="runs/autoencoder_final")

mlflow.set_experiment("autoencoder_cifar10")
with mlflow.start_run(run_name="autoencoder_final"):
    mlflow.log_params({
        "latent_dim": LATENT_DIM, "lr": LR, "epochs": EPOCHS,
        "batch_size": BATCH_SIZE, "lambda_perceptual": LAMBDA_PERCEPTUAL,
    })
    history = train_autoencoder(model, train_loader, optimizer, perceptual_loss_fn,
                                 epochs=EPOCHS, lam=LAMBDA_PERCEPTUAL, writer=writer,
                                 run_name="final")
    mlflow.pytorch.log_model(model, "model")

writer.close()


In [ ]:
# Ver TensorBoard dentro de la notebook
%tensorboard --logdir runs


Para ver la UI de MLflow en Colab hace falta exponer el puerto (por ejemplo con `pyngrok`), ya que Colab no permite acceder directo a `localhost`. Como alternativa simple, después de entrenar pueden descargar la carpeta `mlruns/` y correr `mlflow ui` en su máquina local.

## 7. Visualización de resultados

In [ ]:
model.eval()
with torch.no_grad():
    sample, _ = next(iter(test_loader))
    sample = sample[:8].to(device)
    recon = model(sample)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(sample[i].cpu().permute(1, 2, 0))
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0))
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Reconstruida")
plt.suptitle("Original (arriba) vs Reconstruida (abajo)")
plt.show()


## 8. Preguntas de reflexión (para la defensa)

1. ¿Qué diferencia observaron entre entrenar solo con MSE de píxeles y agregar la perceptual loss? ¿Se nota en las imágenes?
2. ¿Qué pasó con la calidad de reconstrucción al variar `latent_dim`? ¿Por qué creen que ocurre eso?
3. ¿Qué hiperparámetros encontró Optuna como mejores? ¿Tiene sentido con lo que esperaban?
4. Si tuvieran que usar este autoencoder para detectar anomalías (imágenes "raras"), ¿cómo lo usarían?
5. ¿Qué limitación tiene un autoencoder para *generar* imágenes nuevas (no solo reconstruir)? (Esto conecta con la próxima notebook: VAE).
